In [1]:
import os
import sys

# Go to the project root
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

print(project_root)


c:\Users\Haier\Downloads\Senior Data Engineer - Test


In [2]:
from utils.logger import setup_logger
from utils.validation import validate_sales
from utils.exchange_rate import fetch_exchange_rates, create_exchange_rate_df, convert_to_usd
from utils.database import *
from utils.config import *

In [3]:
# Setup
logger = setup_logger()
logger.info("Pipeline Started")

In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Senior DE - ETL")
    .master("local[*]")
    .getOrCreate()
)

## print("Spark Started Successfully!")
# Read Files
sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SALES_FILE)
)

product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCT_FILE)
)

sales_df.printSchema()

product_df.printSchema()

root
 |-- OrderID: integer (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- SaleAmount: double (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Currency: string (nullable = true)

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)



In [5]:
# Cleaning
sales_df = sales_df.dropDuplicates(["OrderID"])
sales_df = sales_df.fillna({"Discount": 0})

# Validation
valid_df, rejected_df = validate_sales(sales_df)




In [6]:
#standarizing date
from pyspark.sql.functions import expr, coalesce

sales_df = sales_df.withColumn(
    "ParsedOrderDate",
    coalesce(
        expr("try_to_timestamp(OrderDate, 'dd/MM/yyyy')"),
        expr("try_to_timestamp(OrderDate, 'dd-MM-yyyy')"),
        expr("try_to_timestamp(OrderDate, 'yyyy-MM-dd')")
    )
)

sales_df = sales_df.drop("OrderDate")

sales_df = sales_df.withColumnRenamed(
    "ParsedOrderDate",
    "OrderDate"
)

In [7]:
# Product Lookup
from pyspark.sql.functions import broadcast

valid_df = valid_df.join(
    broadcast(product_df),
    on="ProductID",
    how="left"
)


In [8]:
logger.info("Reading Data")

In [9]:
valid_df, rejected_df = validate_sales(sales_df)

In [10]:
# Exchange Rates
rates = fetch_exchange_rates()

exchange_df = create_exchange_rate_df(
    spark,
    rates
)

valid_df = convert_to_usd(
    valid_df,
    exchange_df
)

# Display
valid_df.show()

+--------+-------+---------+----------+------+----------+--------+-------------------+------------+------------+-------------+--------------------+
|Currency|OrderID|ProductID|SaleAmount|Region|CustomerID|Discount|          OrderDate|RejectReason|ExchangeRate|SaleAmountUSD| ConversionTimestamp|
+--------+-------+---------+----------+------+----------+--------+-------------------+------------+------------+-------------+--------------------+
|     USD|   1001|      P50|    299.99|  East|      C100|     0.1|2023-05-01 00:00:00|        NULL|         1.0|       299.99|2026-07-05 23:52:...|
|     USD|   1005|      PX1|      89.5| North|      NULL|     0.0|2023-07-01 00:00:00|        NULL|         1.0|         89.5|2026-07-05 23:52:...|
|     GBP|   1007|      P12|     120.0|  East|      C105|     0.1|2023-05-01 00:00:00|        NULL|       0.749|       160.21|2026-07-05 23:52:...|
|     USD|   1008|      P88|     300.0| North|      C106|     0.0|2023-05-02 00:00:00|        NULL|         1.0|

In [11]:
valid_df = valid_df.drop("RejectReason")

In [ ]:
try:
    write_fact_sales(valid_df)
    logger.info("FactSales loaded successfully.")

    write_rejected_records(rejected_df)
    logger.info("RejectedRecords loaded")

    write_currency_log(valid_df)
    logger.info("CurrencyConversionLog loaded")

except Exception as ex:
    import traceback
    traceback.print_exc()
    print("ERROR:", ex)

FactSales loaded
RejectedRecords loaded
CurrencyConversionLog loaded
Pipeline Completed Successfully


In [13]:
logger.info("=" * 60)
logger.info("ETL Pipeline Completed Successfully")
logger.info("=" * 60)

In [15]:
# Pipeline Stats

logger.info(f"Total Records     : {sales_df.count()}")
logger.info(f"Valid Records     : {valid_df.count()}")
logger.info(f"Rejected Records  : {rejected_df.count()}")
